# Embedding-based candidate retrieval (Kaggle GPU)

Fine-tunes a bi-encoder (`sentence-transformers/all-MiniLM-L6-v2`, Apache-2.0) on train (S1, match)
pairs, embeds every record, finds each S1's top-5 nearest S2/S3 records of the same country, and scores
the top-3 with the fine-tuned cross-encoder (attach the cross-encoder notebook's output: `ce_model/`).
Check-set and lockbox S1 are excluded from training, so the check set stays an honest validation.

**Settings:** Accelerator GPU, Internet On. **Inputs:** `amz-er-data` (raw TSVs) and the cross-encoder
notebook's output. **Outputs:** `nn_check.parquet`, `nn_test.parquet` with columns (s1_eid, cand_eid, rank, sim, ce).

In [ ]:
import os, glob, time, math, subprocess, sys, gc
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "anyascii"], check=True)
import numpy as np, pandas as pd, torch, torch.nn.functional as F
from anyascii import anyascii
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification
dev = "cuda" if torch.cuda.is_available() else "cpu"
OUT, TMP = "/kaggle/working", "/kaggle/temp"
os.makedirs(TMP, exist_ok=True)
BI_BASE = "sentence-transformers/all-MiniLM-L6-v2"
N_TRAIN_PAIRS, BATCH, LR, MAXLEN, SCALE = 1_200_000, 384, 5e-5, 64, 20.0
TOPK, CE_RANK = 5, 3
ID_BASE = 10 ** 12
DATA = os.path.dirname(os.path.dirname(sorted(glob.glob("/kaggle/input/**/train/train_source1.tsv", recursive=True))[0]))
REPO = f"{TMP}/amz"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/saiboddapati2k3/amz-ml-challenge.git", REPO], check=True)
check_ids = set(pd.read_parquet(f"{REPO}/artifacts/splits/check_s1.parquet")["eid"].tolist())
lock_ids = set(pd.read_parquet(f"{REPO}/artifacts/splits/lockbox_s1.parquet")["eid"].tolist())
print(dev, torch.cuda.get_device_name(0) if dev == "cuda" else "", DATA, len(check_ids), len(lock_ids))

In [ ]:
def enc_ids(ids):
    s = pd.Series(ids)
    return (s.str.slice(1, 2).astype(np.int64) * ID_BASE + s.str.slice(3).astype(np.int64)).to_numpy()

asc = lambda x: x if x.isascii() else anyascii(x)

def load(split, s):
    d = pd.read_csv(f"{DATA}/{split}/{split}_source{s}.tsv", sep="\t", dtype=str, quoting=3, keep_default_na=False, na_filter=False)
    t = [asc(n) + " ; " + asc(a) for n, a in zip(d["business_name"].values, d["business_address"].values)]
    return pd.DataFrame({"eid": enc_ids(d["entity_id"].values), "country": d["country"].str.strip().values, "t": np.array(t, dtype=object)})

t0 = time.time()
tr1 = load("train", 1); tr23 = pd.concat([load("train", 2), load("train", 3)], ignore_index=True)
te1 = load("test", 1); te23 = pd.concat([load("test", 2), load("test", 3)], ignore_index=True)
print(f"train S1 {len(tr1):,} S2/S3 {len(tr23):,} | test S1 {len(te1):,} S2/S3 {len(te23):,} ({time.time() - t0:.0f}s)")

In [ ]:
# training pairs: one random match per train S1 (no duplicate S1 in a batch), check/lockbox S1 excluded
gt = pd.read_csv(f"{DATA}/train/train_ground_truth.tsv", sep="\t", dtype=str, keep_default_na=False, na_filter=False)
gt = gt[gt["matched_entity_ids"] != ""]
rng = np.random.default_rng(0)
m = [l[rng.integers(len(l))] for l in gt["matched_entity_ids"].str.split(",")]
s1 = enc_ids(gt["source1_entity_id"].values); mb = enc_ids(m)
keep = ~np.isin(s1, np.fromiter(check_ids | lock_ids, dtype=np.int64))
s1, mb = s1[keep], mb[keep]
sel = rng.choice(len(s1), size=min(N_TRAIN_PAIRS, len(s1)), replace=False)
t1 = pd.Series(tr1["t"].values, index=tr1["eid"].values); t23 = pd.Series(tr23["t"].values, index=tr23["eid"].values)
cty = pd.Series(tr1["country"].values, index=tr1["eid"].values)
A, B, C = t1.loc[s1[sel]].values, t23.loc[mb[sel]].values, cty.loc[s1[sel]].values
# same-country batches: in-batch negatives are then plausible distractors
batches = []
for c in np.unique(C):
    idx = rng.permutation(np.flatnonzero(C == c))
    batches += [idx[i:i + BATCH] for i in range(0, len(idx) - BATCH + 1, BATCH)]
rng.shuffle(batches)
print(f"training pairs {len(sel):,}, batches {len(batches):,}")

tok = AutoTokenizer.from_pretrained(BI_BASE)
bi = AutoModel.from_pretrained(BI_BASE).to(dev)

def emb(texts):
    enc = tok(list(texts), padding=True, truncation=True, max_length=MAXLEN, return_tensors="pt").to(dev)
    out = bi(**enc).last_hidden_state
    msk = enc["attention_mask"].unsqueeze(-1).to(out.dtype)
    return F.normalize((out * msk).sum(1) / msk.sum(1).clamp(min=1), dim=-1)

opt = torch.optim.AdamW(bi.parameters(), lr=LR, weight_decay=0.01)
steps, warm = len(batches), max(1, len(batches) // 20)
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / warm) * max(0.0, (steps - s) / max(steps - warm, 1)))
scaler = torch.amp.GradScaler("cuda", enabled=dev == "cuda")
bi.train(); t0 = time.time()
for step, idx in enumerate(batches):
    with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=dev == "cuda"):
        ea, eb = emb(A[idx]), emb(B[idx])
    logits = (ea.float() @ eb.float().T) * SCALE
    lab = torch.arange(len(idx), device=dev)
    loss = (F.cross_entropy(logits, lab) + F.cross_entropy(logits.T, lab)) / 2
    opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt)
    torch.nn.utils.clip_grad_norm_(bi.parameters(), 1.0); scaler.step(opt); scaler.update(); sched.step()
    if step % 500 == 0:
        print(f"step {step}/{steps} loss {loss.item():.4f} ({time.time() - t0:.0f}s)", flush=True)
bi.save_pretrained(f"{OUT}/bi_model"); tok.save_pretrained(f"{OUT}/bi_model")
print(f"bi-encoder trained ({time.time() - t0:.0f}s)")

In [ ]:
@torch.no_grad()
def encode(texts, bs=1024):
    bi.eval()
    order = np.argsort(np.fromiter((len(x) for x in texts), dtype=np.int32, count=len(texts)))
    out = torch.empty((len(texts), bi.config.hidden_size), dtype=torch.float16, device=dev)
    for i in range(0, len(texts), bs):
        idx = order[i:i + bs]
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=dev == "cuda"):
            out[torch.as_tensor(idx, device=dev)] = emb(texts[idx]).half()
    return out

@torch.no_grad()
def knn(Q, P, k=TOPK, qb=1024, pb=1_000_000):
    k = min(k, len(P))
    V = torch.empty((len(Q), k), dtype=torch.float32, device=dev); I = torch.empty((len(Q), k), dtype=torch.int64, device=dev)
    for i in range(0, len(Q), qb):
        q, bv, bi_ = Q[i:i + qb], None, None
        for j in range(0, len(P), pb):
            s = (q @ P[j:j + pb].T).float()
            v, ix = s.topk(min(k, s.shape[1]), dim=1)
            ix = ix + j
            if bv is None:
                bv, bi_ = v, ix
            else:
                v2, pos = torch.cat([bv, v], 1).topk(k, dim=1)
                bi_, bv = torch.cat([bi_, ix], 1).gather(1, pos), v2
        V[i:i + len(q)], I[i:i + len(q)] = bv, bi_
    return V.cpu().numpy(), I.cpu().numpy()

def neighbours(q_df, p_df):
    res = []
    for c in sorted(q_df["country"].unique()):
        qd, pdf = q_df[q_df["country"] == c], p_df[p_df["country"] == c]
        if len(pdf) == 0:
            continue
        t0 = time.time()
        P = encode(pdf["t"].values); Q = encode(qd["t"].values)
        V, I = knn(Q, P)
        k = V.shape[1]
        res.append(pd.DataFrame({"s1_eid": np.repeat(qd["eid"].values, k), "cand_eid": pdf["eid"].values[I.ravel()],
                                 "rank": np.tile(np.arange(k, dtype=np.int8), len(qd)), "sim": V.ravel().astype(np.float32)}))
        del P, Q; torch.cuda.empty_cache()
        print(f"  {c}: {len(qd):,} queries x {len(pdf):,} records ({time.time() - t0:.0f}s)", flush=True)
    return pd.concat(res, ignore_index=True)

nn_check = neighbours(tr1[tr1["eid"].isin(check_ids)], tr23)
nn_test = neighbours(te1, te23)
del bi; gc.collect(); torch.cuda.empty_cache()
print(f"neighbours: check {len(nn_check):,}, test {len(nn_test):,}")

In [ ]:
# cross-encoder on the top-CE_RANK neighbours (the model from the cross-encoder notebook's output)
nn_check["ce"] = np.nan; nn_test["ce"] = np.nan
found = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/ce_model/config.json", recursive=True)]
if not found:
    print("WARNING: no ce_model attached -> cross-encoder scores skipped (add the cross-encoder notebook output as input)")
else:
    ctok = AutoTokenizer.from_pretrained(found[0]); ce = AutoModelForSequenceClassification.from_pretrained(found[0]).to(dev).eval()
    @torch.no_grad()
    def ce_score(a, b, bs=1024):
        order = np.argsort(np.fromiter((len(x) + len(y) for x, y in zip(a, b)), dtype=np.int32, count=len(a)))
        out = np.empty(len(a), dtype=np.float32)
        for i in range(0, len(a), bs):
            idx = order[i:i + bs]
            enc = ctok(list(a[idx]), list(b[idx]), padding=True, truncation=True, max_length=128, return_tensors="pt").to(dev)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=dev == "cuda"):
                out[idx] = torch.sigmoid(ce(**enc).logits.squeeze(-1).float()).cpu().numpy()
        return out
    for name, df, q1, q23 in (("check", nn_check, tr1, tr23), ("test", nn_test, te1, te23)):
        t0 = time.time()
        sub = df.index[df["rank"] < CE_RANK]
        ta = pd.Series(q1["t"].values, index=q1["eid"].values).loc[df.loc[sub, "s1_eid"].values].values
        tb = pd.Series(q23["t"].values, index=q23["eid"].values).loc[df.loc[sub, "cand_eid"].values].values
        df.loc[sub, "ce"] = ce_score(ta, tb)
        print(f"  {name}: {len(sub):,} pairs cross-encoded ({time.time() - t0:.0f}s)", flush=True)
nn_check.to_parquet(f"{OUT}/nn_check.parquet", index=False)
nn_test.to_parquet(f"{OUT}/nn_test.parquet", index=False)
print("wrote", f"{OUT}/nn_check.parquet", f"{OUT}/nn_test.parquet")